# Task 1 — Dataset(s) with exercise-type + quality labels (and your project’s labels)

The referenced paper (arXiv:2202.14019, *Domain Knowledge-Informed Self-Supervised Representations for Workout Form Assessment*) introduces **Fitness-AQA**, which contains multiple exercise types and expert-labeled common form errors. In parallel, this repo already contains real squat videos and knee-error interval annotations.

This notebook documents the label scheme you can derive directly from **your existing project data**, so downstream pose estimation + modeling can run on real videos.


In [ ]:
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

from fitness_adapt.io_utils import load_json
from fitness_adapt.labels import load_error_intervals, compute_quality_labels_for_key
from fitness_adapt.project import ProjectPaths

paths = ProjectPaths.from_root(Path.cwd())
PROJECT_ROOT = paths.root
VIDEO_DIR = paths.squat_video_dir

train_keys = load_json(paths.split_path('train'))
val_keys = load_json(paths.split_path('val'))
test_keys = load_json(paths.split_path('test'))

error_fwd, error_inward = load_error_intervals(paths)

def get_video_duration_sec(video_path: Path) -> float:
    cap = cv2.VideoCapture(str(video_path))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return frames / fps if fps > 0 else 0.0

def derive_labels(video_key: str, *, exercise_type: str = 'squat'):
    video_path = VIDEO_DIR / f'{video_key}.mp4'
    duration_sec = get_video_duration_sec(video_path)
    return compute_quality_labels_for_key(
        video_key,
        exercise_type=exercise_type,
        duration_sec=duration_sec,
        error_fwd=error_fwd,
        error_inward=error_inward,
    )


In [ ]:
# Quick sanity check on a handful of keys
sample_keys = train_keys[:8]

rows = []
for k in sample_keys:
    labels = derive_labels(k)
    rows.append((k, labels.exercise_type, labels.quality_binary, labels.quality_score))

for r in rows:
    print(r)


## What “two types of labels” means in your current project

- **Exercise type label**: currently constant = `squat` (your repo only ships squat videos).
- **Quality label**: derived from the provided annotations:
  - `error_knees_forward.json` contains time intervals where the knee moves too far forward.
  - `error_knees_inward.json` contains time intervals where the knee collapses inward.

We derive:
- `quality_binary = 1` if no annotated error intervals exist; else `0`.
- `quality_score = 1 - (union_error_coverage / duration)` in `[0,1]`.

If you later add more exercises (lunge, push-up, etc.), you can keep the same logic—only the exercise-type label would change per video key.
